# Connecting Node.js to MongoDB with Mongoose

You connect a Node.js application to MongoDB by calling `mongoose.connect()` from the [Mongoose ODM](https://mongoosejs.com/) library.

```bash
npm install mongoose
```

## Basic connection

```javascript
const mongoose = require('mongoose');

async function main() {
  await mongoose.connect('mongodb://127.0.0.1:27017/test');
  console.log('Connected to MongoDB');
}

main().catch(err => console.log(err));
```

ESM equivalent:

```javascript
import mongoose from 'mongoose';

try {
  await mongoose.connect('mongodb://127.0.0.1:27017/test');
  console.log('Connected to MongoDB');
} catch (err) {
  console.log(err);
}
```

## Anatomy of the connection string

```
mongodb://127.0.0.1:27017/test?retryWrites=true
└──┬───┘  └────┬────┘ └─┬─┘ └┬─┘ └──────┬─────┘
protocol    host      port  db      options
```

| Part | Notes |
|---|---|
| `mongodb://` | Standard protocol for a local or self-hosted server |
| `mongodb+srv://` | DNS seed-list format used by Atlas and replica sets; no port |
| host | `127.0.0.1` locally, cluster hostname on Atlas |
| port | `27017` is the default; omitted with `+srv` |
| db name | Created lazily — Mongo won't materialise it until you write data |
| options | Query-string pairs, e.g. `retryWrites`, `w=majority`, `authSource` |

Atlas / hosted example:

```
mongodb+srv://<user>:<password>@cluster0.abcde.mongodb.net/myApp?retryWrites=true&w=majority
```

Never commit a URI with credentials. Keep it in `.env` and load it:

```javascript
require('dotenv').config();
await mongoose.connect(process.env.MONGO_URI);
```

If the password contains `@`, `:`, `/` or `%`, percent-encode it with `encodeURIComponent()`.

## Essential guidelines

- **Localhost address** — Use `127.0.0.1` instead of `localhost`. Modern Node.js resolves `localhost` to IPv6 (`::1`) first, and a MongoDB server bound only to IPv4 will refuse that connection.
- **Command buffering** — Mongoose buffers model function calls until the connection opens, so you can define models and fire queries before `connect()` resolves. Disable with `{ bufferCommands: false }` if you'd rather fail fast. Buffered operations time out after `bufferTimeoutMS` (default 10000).
- **Multiple connections** — Use `mongoose.createConnection()` when the app talks to multiple databases or clusters. It returns a `Connection` you register models on directly (`conn.model('User', schema)`) rather than the global `mongoose.model()`.
- **Error handling** — Wrap the connection call in an async function with `.catch()` or `try/catch`. This only catches the *initial* failure; errors after a successful connect surface on the connection's event emitter (below).
- **Connect once, at startup** — `mongoose.connect()` opens a pooled connection meant to live for the process lifetime. Don't call it per request or per route handler.

## Useful options

```javascript
await mongoose.connect(process.env.MONGO_URI, {
  serverSelectionTimeoutMS: 5000, // fail fast instead of waiting 30s (default)
  socketTimeoutMS: 45000,         // close sockets after inactivity
  maxPoolSize: 10,                // default 100; tune for your workload
  dbName: 'myApp',                // overrides the db in the URI
  autoIndex: false                // disable in production; build indexes deliberately
});
```

> **Version note:** `useNewUrlParser`, `useUnifiedTopology`, `useCreateIndex` and `useFindAndModify` were removed in Mongoose 6/7. Passing them in Mongoose 7+ is a no-op or throws. If a tutorial includes them, it predates v6.

## Connection events

Initial-connection errors reject the promise. Everything after that — dropped network, replica-set failover, auth revoked — arrives as an event:

```javascript
const db = mongoose.connection;

db.on('error', err => console.error('MongoDB error:', err));
db.once('open', () => console.log('MongoDB connection open'));
db.on('disconnected', () => console.warn('MongoDB disconnected'));
db.on('reconnected', () => console.log('MongoDB reconnected'));
```

Mongoose retries automatically after a disconnect, so you generally log rather than exit.

`mongoose.connection.readyState` maps to: `0` disconnected, `1` connected, `2` connecting, `3` disconnecting.

## Wiring it into Express

Connect before the server starts listening, so the app never accepts traffic it can't serve:

```javascript
const express = require('express');
const mongoose = require('mongoose');

const app = express();

async function start() {
  await mongoose.connect(process.env.MONGO_URI);
  console.log('Connected to MongoDB');
  app.listen(3000, () => console.log('Listening on 3000'));
}

start().catch(err => {
  console.error('Startup failed:', err);
  process.exit(1);
});
```

## Closing the connection

```javascript
await mongoose.disconnect();
```

Graceful shutdown:

```javascript
process.on('SIGINT', async () => {
  await mongoose.connection.close();
  process.exit(0);
});
```

Needed for scripts and seed files (otherwise the process hangs on the open socket) and in test teardown.

## Common errors

| Error | Usual cause |
|---|---|
| `ECONNREFUSED 127.0.0.1:27017` | `mongod` isn't running, or you're hitting `localhost` → `::1` |
| `MongooseServerSelectionError` | Wrong host, firewall, or IP not allowlisted in Atlas |
| `Operation "x.find()" buffering timed out after 10000ms` | Query ran but `connect()` never succeeded |
| `MongoServerError: bad auth` | Wrong credentials, or wrong `authSource` database |
| `MongoParseError: option ... is not supported` | Legacy `useNewUrlParser`-era options on Mongoose 7+ |

## Sources

- [Mongoose docs](https://mongoosejs.com/docs/)
- [Connections guide](https://mongoosejs.com/docs/connections.html)
- [mongoose on npm](https://www.npmjs.com/package/mongoose)